# AlexNet Training
Adapted from the original AlexNet model by Krizhevsky, et al. A more modern approach of this model is to use Batch Normalisation between layers instead of the original Local Response Normalisation

*Dataset:*
- 500 class subset of iNaturalist-2021 (mini)
- 50 images per class (40:10 train-test split)
- Data samples processed in src/data_processing/sampling_dataset.ipynb


## Device Check

In this stage, we import the necessary libraries and confirm the torch device used

In [13]:
from pathlib import Path
import json

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import alexnet, AlexNet, AlexNet_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device: ", device)

if torch.cuda.is_available():
    print("GPU: ", torch.cuda.get_device_name(0))

Device:  cuda
GPU:  NVIDIA L4


## Config
Establish paths

In [14]:
DATA_ROOT = Path(r"/home/sergiocloudwork/dataset/sampled_500")

TRAIN_DIR = DATA_ROOT / "train_mini"
VAL_DIR = DATA_ROOT / "validation"

MODEL_DIR = DATA_ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)


## Processing
Enable transforms to process images to tensors for model input.
While the original AlexNet takes image inputs 224x224x3, we expanded the dimensions to 320x320x3 to ensure details in large-sized images are not lost to downsampling

In [15]:

INPUT_SIZE = 320
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.CenterCrop(INPUT_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(
        brightness=0.10,
        contrast=0.10,
        saturation=0.10
    ),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

val_transform = transforms.Compose([
    transforms.CenterCrop(INPUT_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

## Load Training and Validation
Load the datasets that will be used for the models

In [16]:
train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)

val_dataset = datasets.ImageFolder(VAL_DIR, transform=val_transform)

assert train_dataset.class_to_idx == val_dataset.class_to_idx, (
    "Train and validation folder differed"
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=8,
    pin_memory = torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=8,
    pin_memory=torch.cuda.is_available()
)

print("Classes:", len(train_dataset.classes))
print("Train images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Class mapping:", train_dataset.class_to_idx)

Classes: 500
Train images: 20000
Validation images: 5000
Class mapping: {'00001_Animalia_Annelida_Polychaeta_Sabellida_Sabellidae_Sabella_spallanzanii': 0, '00003_Animalia_Annelida_Polychaeta_Sabellida_Serpulidae_Spirobranchus_cariniferus': 1, '00018_Animalia_Arthropoda_Arachnida_Araneae_Araneidae_Argiope_bruennichi': 2, '00024_Animalia_Arthropoda_Arachnida_Araneae_Araneidae_Cyclosa_turbinata': 3, '00038_Animalia_Arthropoda_Arachnida_Araneae_Araneidae_Neoscona_crucifera': 4, '00055_Animalia_Arthropoda_Arachnida_Araneae_Filistatidae_Kukulcania_hibernalis': 5, '00128_Animalia_Arthropoda_Arachnida_Araneae_Thomisidae_Synema_globosum': 6, '00145_Animalia_Arthropoda_Arachnida_Opiliones_Phalangiidae_Phalangium_opilio': 7, '00159_Animalia_Arthropoda_Chilopoda_Scolopendromorpha_Scolopendridae_Scolopendra_heros': 8, '00203_Animalia_Arthropoda_Insecta_Coleoptera_Carabidae_Cicindela_hirticollis': 9, '00216_Animalia_Arthropoda_Insecta_Coleoptera_Carabidae_Scaphinotus_angusticollis': 10, '00230_Anim

## Training Function
Establish the training process of the model

In [17]:
def training(
    EPOCHS,
    model,
    optimizer,
    scheduler,
    criterion,
    fname,
    classes
):
    best_val_accuracy = -1.0
    early_stop_patience = 7
    epochs_without_improvement = 0
    min_improvement = 1e-4

    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    # save training
    history = {
    "epoch": [],
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": [],
    "learning_rates": []
    }
    history_fname = str(Path(fname).with_suffix("")) + "_history.json"

    for epoch in range(EPOCHS):
        #  Training 
        model.train()

        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for images, labels in train_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=use_amp
            ):
                outputs = model(images)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item() * labels.size(0)
            train_correct += (
                outputs.argmax(dim=1) == labels
            ).sum().item()
            train_total += labels.size(0)

        #  Validation 
        model.eval()

        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.inference_mode():
            for images, labels in val_loader:
                images = images.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)

                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16,
                    enabled=use_amp
                ):
                    outputs = model(images)
                    loss = criterion(outputs, labels)

                val_loss += loss.item() * labels.size(0)
                val_correct += (
                    outputs.argmax(dim=1) == labels
                ).sum().item()
                val_total += labels.size(0)

        train_loss /= train_total
        val_loss /= val_total
        train_accuracy = train_correct / train_total
        val_accuracy = val_correct / val_total

        scheduler.step(val_accuracy)

        learning_rates = [
            f"{group['lr']:.2e}"
            for group in optimizer.param_groups
        ]

        print(
            f"Epoch {epoch + 1:02d}/{EPOCHS} | "
            f"Train loss: {train_loss:.4f} | "
            f"Train acc: {train_accuracy:.4f} | "
            f"Val loss: {val_loss:.4f} | "
            f"Val acc: {val_accuracy:.4f} | "
            f"LR: {learning_rates}"
        )
        
        # data adding
        history["epoch"].append(epoch + 1)
        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_accuracy)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_accuracy)
        history["learning_rates"].append(
            [group["lr"] for group in optimizer.param_groups]
        )
        with open(history_fname, "w", encoding="utf-8") as f:
            json.dump(history, f, indent=4)

        if val_accuracy > best_val_accuracy + min_improvement:
            best_val_accuracy = val_accuracy
            epochs_without_improvement = 0

            torch.save({
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "classes": classes,
                "num_classes": len(classes),
                "best_val_accuracy": best_val_accuracy,
                "train_accuracy": train_accuracy,
                "val_accuracy": val_accuracy,
                "train_loss": train_loss,
                "val_loss": val_loss
            }, (fname+".pth"))

            print(
                f"Best model saved: {(fname+'.pth')} "
                f"(val_acc={best_val_accuracy:.4f})"
            )

        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= early_stop_patience:
            print(
                f"Early stopping. "
                f"Best val accuracy: {best_val_accuracy:.4f}"
            )
            break

    return best_val_accuracy

## Load Pre-Trained Model
Load the pre-trained model and run the training function

In [18]:
weights = AlexNet_Weights.IMAGENET1K_V1

pt_model = alexnet(weights=weights)
num_classes = len(train_dataset.classes)
in_features = pt_model.classifier[6].in_features

pt_model.classifier[6] = nn.Linear(in_features, num_classes)

# Conv1+pool, Conv2+pool
early_parameters = [
    parameter
    for module in (
        pt_model.features[0:6],
    )
    for parameter in module.parameters()
]

# Conv3, Conv4, Conv5+pool
late_parameters = [
    parameter
    for module in (
        pt_model.features[6:13],
    )
    for parameter in module.parameters()
]

pt_model = pt_model.to(device)
pt_criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
pt_optimizer = torch.optim.AdamW([
    {"params": early_parameters, "lr": 3e-6},
    {"params": late_parameters, "lr": 1e-5},
    {"params": pt_model.classifier.parameters(), "lr": 1e-4},
], weight_decay=5e-4)

pt_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    pt_optimizer,
    mode="max",
    factor=0.5,
    patience=3,
    threshold=0.0005,
    threshold_mode="rel",
    cooldown=1,
    min_lr=1e-7
)

num_classes = len(train_dataset.classes)
pt_model_filename = f"inat_alexnet_{num_classes}_ptclasses_imagenet"
training(50,pt_model,pt_optimizer,pt_scheduler,pt_criterion,pt_model_filename,train_dataset.classes)


Epoch 01/50 | Train loss: 4.2687 | Train acc: 0.2464 | Val loss: 3.3750 | Val acc: 0.4074 | LR: ['3.00e-06', '1.00e-05', '1.00e-04']
Best model saved: inat_alexnet_500_ptclasses_imagenet.pth (val_acc=0.4074)
Epoch 02/50 | Train loss: 2.7969 | Train acc: 0.5363 | Val loss: 3.1592 | Val acc: 0.4542 | LR: ['3.00e-06', '1.00e-05', '1.00e-04']
Best model saved: inat_alexnet_500_ptclasses_imagenet.pth (val_acc=0.4542)
Epoch 03/50 | Train loss: 2.2596 | Train acc: 0.6884 | Val loss: 3.1105 | Val acc: 0.4660 | LR: ['3.00e-06', '1.00e-05', '1.00e-04']
Best model saved: inat_alexnet_500_ptclasses_imagenet.pth (val_acc=0.4660)
Epoch 04/50 | Train loss: 1.8733 | Train acc: 0.8064 | Val loss: 3.1430 | Val acc: 0.4740 | LR: ['3.00e-06', '1.00e-05', '1.00e-04']
Best model saved: inat_alexnet_500_ptclasses_imagenet.pth (val_acc=0.4740)
Epoch 05/50 | Train loss: 1.6286 | Train acc: 0.8867 | Val loss: 3.1529 | Val acc: 0.4766 | LR: ['3.00e-06', '1.00e-05', '1.00e-04']
Best model saved: inat_alexnet_500_

0.5124

## Train with No Preset Weights
Train model with no established weights

In [19]:
np_model = alexnet(weights=None)
num_classes = len(train_dataset.classes)
in_features = np_model.classifier[6].in_features

np_model.classifier[6] = nn.Linear(in_features, num_classes)

np_model = np_model.to(device)
np_criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

np_optimizer = torch.optim.AdamW(
    np_model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

np_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    np_optimizer,
    mode="max",
    factor=0.5,
    patience=2,
    threshold=1e-3,
    min_lr=1e-7
)

num_classes = len(train_dataset.classes)
np_model_filename = f"inat_alexnet_{num_classes}_npclasses_imagenet"

training(50, np_model, np_optimizer, np_scheduler, np_criterion, np_model_filename, train_dataset.classes)

Epoch 01/50 | Train loss: 6.1680 | Train acc: 0.0022 | Val loss: 6.0418 | Val acc: 0.0042 | LR: ['1.00e-04']
Best model saved: inat_alexnet_500_npclasses_imagenet.pth (val_acc=0.0042)
Epoch 02/50 | Train loss: 5.9635 | Train acc: 0.0062 | Val loss: 5.8253 | Val acc: 0.0112 | LR: ['1.00e-04']
Best model saved: inat_alexnet_500_npclasses_imagenet.pth (val_acc=0.0112)
Epoch 03/50 | Train loss: 5.7284 | Train acc: 0.0128 | Val loss: 5.6345 | Val acc: 0.0216 | LR: ['1.00e-04']
Best model saved: inat_alexnet_500_npclasses_imagenet.pth (val_acc=0.0216)
Epoch 04/50 | Train loss: 5.5533 | Train acc: 0.0264 | Val loss: 5.5179 | Val acc: 0.0310 | LR: ['1.00e-04']
Best model saved: inat_alexnet_500_npclasses_imagenet.pth (val_acc=0.0310)
Epoch 05/50 | Train loss: 5.4014 | Train acc: 0.0359 | Val loss: 5.3812 | Val acc: 0.0454 | LR: ['1.00e-04']
Best model saved: inat_alexnet_500_npclasses_imagenet.pth (val_acc=0.0454)
Epoch 06/50 | Train loss: 5.2491 | Train acc: 0.0502 | Val loss: 5.2647 | Val ac

0.173

## Model Reloading
Show the final realised model and resume training

In [22]:
pt_model_filename= Path("inat_alexnet_500_ptclasses_imagenet.pth")
ADDITIONAL_EPOCHS = 20   
LR_FACTOR = 0.3          

checkpoint = torch.load(
    pt_model_filename,
    map_location=device,
    weights_only=True
)


assert checkpoint["classes"] == train_dataset.classes
assert checkpoint["num_classes"] == len(train_dataset.classes)

# recover
pt_model.load_state_dict(
    checkpoint["model_state_dict"]
)

pt_optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

pt_scheduler.load_state_dict(
    checkpoint["scheduler_state_dict"]
)

# make sure optizmizer is right device
for state in pt_optimizer.state.values():
    for key, value in state.items():
        if torch.is_tensor(value):
            state[key] = value.to(device)

start_epoch = checkpoint["epoch"]
best_val_accuracy = checkpoint["best_val_accuracy"]


for group in pt_optimizer.param_groups:
    group["lr"] *= LR_FACTOR

total_epochs = start_epoch + ADDITIONAL_EPOCHS

print(f"Loaded: {pt_model_filename}")
print(f"Resume from epoch: {start_epoch + 1}")
print(f"Train until epoch: {total_epochs}")
print(f"Previous best val accuracy: {best_val_accuracy:.4f}")
print("Learning rates:", [
    group["lr"]
    for group in pt_optimizer.param_groups
])

best_accuracy = training(
    total_epochs,
    pt_model,
    pt_optimizer,
    pt_scheduler,
    pt_criterion,
    pt_model_filename,
    train_dataset.classes,
)

Loaded: inat_alexnet_500_ptclasses_imagenet.pth
Resume from epoch: 24
Train until epoch: 43
Previous best val accuracy: 0.5124
Learning rates: [2.25e-07, 7.5e-07, 7.5e-06]
Epoch 01/43 | Train loss: 1.1018 | Train acc: 0.9998 | Val loss: 3.2089 | Val acc: 0.5136 | LR: ['2.25e-07', '7.50e-07', '7.50e-06']


TypeError: unsupported operand type(s) for +: 'PosixPath' and 'str'